<a href="https://colab.research.google.com/github/MagimaaSree2007/numpy-deep-learning-stack/blob/main/Robot_ACE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# Load Fashion-MNIST dataset using Keras (standard pre-installed utility in Colab)
from tensorflow.keras.datasets import fashion_mnist

(X_train_raw, y_train), (X_test_raw, y_test) = fashion_mnist.load_data()

# Normalize pixel values to [0, 1] and flatten for MLP if needed, keep 2D for CNN
X_train = X_train_raw.astype(np.float32) / 255.0
X_test = X_test_raw.astype(np.float32) / 255.0

print("Data loaded successfully!")
print(f"Training shapes -> Images: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test shapes     -> Images: {X_test.shape}, Labels: {y_test.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Data loaded successfully!
Training shapes -> Images: (60000, 28, 28), Labels: (60000,)
Test shapes     -> Images: (10000, 28, 28), Labels: (10000,)


In [2]:
class Tensor:

  def __init__(self, data, parents=(), creator=""):
    self.data = np.array(data, dtype=np.float32)
    self.grad = np.zeros_like(self.data)
    self.parents = parents
    self.creator = creator
    self.grad_fn = lambda: None

  def __add__(self, other):
    other = other if isinstance(other, Tensor) else Tensor(other)
    out = Tensor(self.data + other.data, parents=(self, other), creator="add")

    def grad_fn():
      self.grad += unbroadcast(out.grad, self.data.shape)
      other.grad += unbroadcast(out.grad, other.data.shape)

    out.grad_fn = grad_fn
    return out

  def __matmul__(self, other):
    out = Tensor(
        self.data @ other.data, parents=(self, other), creator="matmul"
    )

    def grad_fn():
      self.grad += out.grad @ other.data.T
      other.grad += self.data.T @ out.grad

    out.grad_fn = grad_fn
    return out

  def relu(self):
    out = Tensor(np.maximum(0, self.data), parents=(self,), creator="relu")

    def grad_fn():
      self.grad += out.grad * (self.data > 0)

    out.grad_fn = grad_fn
    return out

  def backward(self):
    # Topological sort
    topo = []
    visited = set()

    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for p in v.parents:
          build_topo(p)
        topo.append(v)

    build_topo(self)
    self.grad = np.ones_like(self.data)
    for v in reversed(topo):
      v.grad_fn()


def unbroadcast(grad, target_shape):
  if grad.shape == target_shape:
    return grad
  axis_diff = len(grad.shape) - len(target_shape)
  for i in range(axis_diff):
    grad = grad.sum(axis=0)
  for i, dim in enumerate(target_shape):
    if dim == 1:
      grad = grad.sum(axis=i, keepdims=True)
  return grad

In [3]:
def softmax_cross_entropy(logits, y):
  # Numerical stability shift
  shift_logits = logits.data - np.max(logits.data, axis=1, keepdims=True)
  exps = np.exp(shift_logits)
  probs = exps / np.sum(exps, axis=1, keepdims=True)

  N = logits.data.shape[0]
  loss_val = -np.mean(np.log(probs[np.arange(N), y] + 1e-9))

  loss = Tensor(loss_val, parents=(logits,), creator="ce_loss")

  def grad_fn():
    dlogits = probs.copy()
    dlogits[np.arange(N), y] -= 1
    dlogits /= N
    logits.grad += dlogits * loss.grad

  loss.grad_fn = grad_fn
  return loss, probs


class Linear:

  def __init__(self, in_features, out_features):
    limit = np.sqrt(2.0 / in_features)
    self.W = Tensor(np.random.randn(in_features, out_features) * limit)
    self.b = Tensor(np.zeros((1, out_features)))

  def __call__(self, x):
    return x @ self.W + self.b

  @property
  def params(self):
    return [self.W, self.b]


class Adam:

  def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
    self.params = params
    self.lr = lr
    self.beta1 = beta1
    self.beta2 = beta2
    self.eps = eps
    self.m = [np.zeros_like(p.data) for p in params]
    self.v = [np.zeros_like(p.data) for p in params]
    self.t = 0

  def step(self):
    self.t += 1
    for i, p in enumerate(self.params):
      if p.grad is None:
        continue
      self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * p.grad
      self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (p.grad**2)
      m_hat = self.m[i] / (1 - self.beta1**self.t)
      v_hat = self.v[i] / (1 - self.beta2**self.t)
      p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

In [4]:
def get_im2col_indices(x_shape, field_height, field_width, padding=1, stride=1):
  N, C, H, W = x_shape
  out_height = int((H + 2 * padding - field_height) / stride + 1)
  out_width = int((W + 2 * padding - field_width) / stride + 1)

  i0 = np.repeat(np.arange(field_height), field_width)
  i0 = np.tile(i0, C)
  i_stride = stride * np.repeat(np.arange(out_height), out_width)
  j0 = np.tile(np.arange(field_width), field_height * C)
  j_stride = np.tile(stride * np.arange(out_width), out_height)

  i = i0.reshape(-1, 1) + i_stride.reshape(1, -1)
  j = j0.reshape(-1, 1) + j_stride.reshape(1, -1)
  k = np.repeat(np.arange(C), field_height * field_width).reshape(-1, 1)
  return (k.astype(int), i.astype(int), j.astype(int))


def im2col_indices(x, field_height, field_width, padding=1, stride=1):
  p = padding
  x_padded = np.pad(
      x,
      ((0, 0), (0, 0), (p, p), (p, p)),
      mode="constant",
      constant_values=0,
  )
  k, i, j = get_im2col_indices(
      x.shape, field_height, field_width, padding, stride
  )
  cols = x_padded[:, k, i, j]
  C = x.shape[1]
  cols = cols.transpose(1, 2, 0).reshape(field_height * field_width * C, -1)
  return cols


def col2im_indices(
    cols, x_shape, field_height=3, field_width=3, padding=1, stride=1
):
  N, C, H, W = x_shape
  H_padded, W_padded = H + 2 * padding, W + 2 * padding
  x_padded = np.zeros((N, C, H_padded, W_padded), dtype=cols.dtype)
  k, i, j = get_im2col_indices(
      x_shape, field_height, field_width, padding, stride
  )
  cols_reshaped = cols.reshape(C * field_height * field_width, -1, N)
  cols_reshaped = cols_reshaped.transpose(2, 0, 1)
  np.add.at(x_padded, (slice(None), k, i, j), cols_reshaped)
  if padding == 0:
    return x_padded
  return x_padded[:, :, padding:-padding, padding:-padding]


class Conv2D:

  def __init__(
      self,
      in_channels,
      out_channels,
      kernel_size=3,
      stride=1,
      padding=1,
  ):
    self.ks = kernel_size
    self.stride = stride
    self.padding = padding
    limit = np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
    self.W = Tensor(
        np.random.randn(out_channels, in_channels, kernel_size, kernel_size)
        * limit
    )
    self.b = Tensor(np.zeros((out_channels, 1)))

  def __call__(self, x):
    N, C, H, W = x.data.shape
    out_h = int((H + 2 * self.padding - self.ks) / self.stride) + 1
    out_w = int((W + 2 * self.padding - self.ks) / self.stride) + 1

    X_col = im2col_indices(
        x.data, self.ks, self.ks, padding=self.padding, stride=self.stride
    )
    W_row = self.W.data.reshape(self.W.data.shape[0], -1)

    out_data = (W_row @ X_col + self.b.data).reshape(
        self.W.data.shape[0], out_h, out_w, N
    )
    out_data = out_data.transpose(3, 0, 1, 2)

    out = Tensor(out_data, parents=(x, self.W, self.b), creator="conv2d")

    def grad_fn():
      dout = out.grad
      dout_reshaped = dout.transpose(1, 2, 3, 0).reshape(
          self.W.data.shape[0], -1
      )
      self.W.grad += (dout_reshaped @ X_col.T).reshape(self.W.data.shape)
      self.b.grad += np.sum(dout_reshaped, axis=1, keepdims=True)
      W_row = self.W.data.reshape(self.W.data.shape[0], -1)
      dX_col = W_row.T @ dout_reshaped
      dx = col2im_indices(
          dX_col,
          x.data.shape,
          self.ks,
          self.ks,
          padding=self.padding,
          stride=self.stride,
      )
      x.grad += dx

    out.grad_fn = grad_fn
    return out

  @property
  def params(self):
    return [self.W, self.b]


class MaxPool:

  def __init__(self, size=2, stride=2):
    self.h = size
    self.w = size
    self.stride = stride

  def __call__(self, x):
    N, C, H, W = x.data.shape
    out_h = int((H - self.h) / self.stride + 1)
    out_w = int((W - self.w) / self.stride + 1)

    x_reshaped = x.data.reshape(N * C, 1, H, W)
    X_col = im2col_indices(
        x_reshaped, self.h, self.w, padding=0, stride=self.stride
    )

    max_idx = np.argmax(X_col, axis=0)
    out = X_col[max_idx, np.arange(max_idx.size)]
    out = out.reshape(out_h, out_w, N, C).transpose(2, 3, 0, 1)

    self.cache = (x.data.shape, X_col, max_idx, self.h, self.w, self.stride)
    out_tensor = Tensor(out, parents=(x,), creator="maxpool")

    def grad_fn():
      x_shape, X_col, max_idx, h, w, stride = self.cache
      dout = out_tensor.grad.transpose(2, 3, 0, 1).ravel()
      dX_col = np.zeros_like(X_col)
      dX_col[max_idx, np.arange(max_idx.size)] = dout
      dx = col2im_indices(dX_col, x_shape, h, w, padding=0, stride=stride)
      x.grad += dx

    out_tensor.grad_fn = grad_fn
    return out_tensor


class BatchNorm:

  def __init__(self, channels, momentum=0.9, eps=1e-5):
    self.gamma = Tensor(np.ones((1, channels, 1, 1)))
    self.beta = Tensor(np.zeros((1, channels, 1, 1)))
    self.momentum = momentum
    self.eps = eps
    self.running_mean = np.zeros((1, channels, 1, 1))
    self.running_var = np.ones((1, channels, 1, 1))
    self.train_mode = True

  def __call__(self, x):
    if self.train_mode:
      mean = np.mean(x.data, axis=(0, 2, 3), keepdims=True)
      var = np.var(x.data, axis=(0, 2, 3), keepdims=True)
      self.running_mean = (
          self.momentum * self.running_mean + (1 - self.momentum) * mean
      )
      self.running_var = (
          self.momentum * self.running_var + (1 - self.momentum) * var
      )

      x_hat = (x.data - mean) / np.sqrt(var + self.eps)
      out_data = self.gamma.data * x_hat + self.beta.data
      out = Tensor(
          out_data, parents=(x, self.gamma, self.beta), creator="batchnorm"
      )

      def grad_fn():
        dout = out.grad
        self.gamma.grad += np.sum(dout * x_hat, axis=(0, 2, 3), keepdims=True)
        self.beta.grad += np.sum(dout, axis=(0, 2, 3), keepdims=True)
        dx = (
            (self.gamma.data / np.sqrt(var + self.eps))
            * (
                dout
                - np.mean(dout, axis=(0, 2, 3), keepdims=True)
                - x_hat
                * np.mean(dout * x_hat, axis=(0, 2, 3), keepdims=True)
            )
        )
        x.grad += dx

      out.grad_fn = grad_fn
      return out
    else:
      x_hat = (x.data - self.running_mean) / np.sqrt(
          self.running_var + self.eps
      )
      out_data = self.gamma.data * x_hat + self.beta.data
      return Tensor(out_data)

  @property
  def params(self):
    return [self.gamma, self.beta]


class Dropout:

  def __init__(self, p=0.5):
    self.p = p
    self.train_mode = True

  def __call__(self, x):
    if self.train_mode and self.p > 0:
      mask = (np.random.rand(*x.data.shape) >= self.p) / (1.0 - self.p)
      out_data = x.data * mask
      out = Tensor(out_data, parents=(x,), creator="dropout")

      def grad_fn():
        x.grad += out.grad * mask

      out.grad_fn = grad_fn
      return out
    else:
      return x

In [5]:
class SimpleCNN:

  def __init__(self):
    self.conv1 = Conv2D(
        in_channels=1, out_channels=16, kernel_size=3, padding=1
    )
    self.bn1 = BatchNorm(channels=16)
    self.pool1 = MaxPool(size=2, stride=2)

    self.conv2 = Conv2D(
        in_channels=16, out_channels=32, kernel_size=3, padding=1
    )
    self.bn2 = BatchNorm(channels=32)
    self.pool2 = MaxPool(size=2, stride=2)

    self.fc1 = Linear(32 * 7 * 7, 128)
    self.dropout = Dropout(p=0.3)
    self.fc2 = Linear(128, 10)

  def set_train_mode(self, mode):
    self.bn1.train_mode = mode
    self.bn2.train_mode = mode
    self.dropout.train_mode = mode

  def __call__(self, x):
    out = self.conv1(x)
    out = self.bn1(out)
    out = out.relu()
    out = self.pool1(out)

    out = self.conv2(out)
    out = self.bn2(out)
    out = out.relu()
    pool_out = self.pool2(out)

    N = pool_out.data.shape[0]
    out_flat = Tensor(
        pool_out.data.reshape(N, -1), parents=(pool_out,), creator="flatten"
    )

    def grad_fn():
      pool_out.grad += out_flat.grad.reshape(pool_out.data.shape)

    out_flat.grad_fn = grad_fn

    fc_out = self.fc1(out_flat).relu()
    fc_out = self.dropout(fc_out)
    logits = self.fc2(fc_out)
    return logits

  @property
  def params(self):
    return (
        self.conv1.params
        + self.bn1.params
        + self.conv2.params
        + self.bn2.params
        + self.fc1.params
        + self.fc2.params
    )


# Prepare CNN format shapes
X_train_cnn = X_train.reshape(-1, 1, 28, 28)
X_test_cnn = X_test.reshape(-1, 1, 28, 28)

# Initialize & Train
np.random.seed(42)
cnn_model = SimpleCNN()
optimizer = Adam(cnn_model.params, lr=0.001)

batch_size = 64
epochs = 15

print("Starting CNN Training...")
for epoch in range(epochs):
  cnn_model.set_train_mode(True)
  indices = np.arange(X_train_cnn.shape[0])
  np.random.shuffle(indices)
  X_shuff = X_train_cnn[indices]
  y_shuff = y_train[indices]

  total_loss = 0
  correct = 0

  for i in range(0, len(X_train_cnn), batch_size):
    xb = Tensor(X_shuff[i : i + batch_size])
    yb = y_shuff[i : i + batch_size]

    logits = cnn_model(xb)
    loss, probs = softmax_cross_entropy(logits, yb)

    for p in cnn_model.params:
      p.grad = np.zeros_like(p.data)

    loss.backward()
    optimizer.step()

    total_loss += loss.data * len(yb)
    preds = np.argmax(probs, axis=1)
    correct += np.sum(preds == yb)

  train_acc = correct / len(X_train_cnn)
  print(
      f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(X_train_cnn):.4f} |"
      f" Train Acc: {train_acc*100:.2f}%"
  )

# Evaluate on Test Set
cnn_model.set_train_mode(False)
test_correct = 0

print("\nRunning final CNN test evaluation...")
for i in range(0, len(X_test_cnn), batch_size):
  xb = Tensor(X_test_cnn[i : i + batch_size])
  yb = y_test[i : i + batch_size]

  logits = cnn_model(xb)
  exp_probs = np.exp(logits.data - np.max(logits.data, axis=1, keepdims=True))
  probs = exp_probs / np.sum(exp_probs, axis=1, keepdims=True)

  preds = np.argmax(probs, axis=1)
  test_correct += np.sum(preds == yb)

cnn_test_acc = (test_correct / len(X_test_cnn)) * 100
print(f"Final CNN Test Accuracy: {cnn_test_acc:.2f}% (Target: >= 91%)[cite: 1]")

Starting CNN Training...
Epoch 1/15 | Loss: 0.5309 | Train Acc: 80.79%
Epoch 2/15 | Loss: 0.4106 | Train Acc: 85.03%
Epoch 3/15 | Loss: 0.3944 | Train Acc: 85.51%
Epoch 4/15 | Loss: 0.3908 | Train Acc: 85.62%
Epoch 5/15 | Loss: 0.3838 | Train Acc: 85.73%
Epoch 6/15 | Loss: 0.3845 | Train Acc: 85.94%
Epoch 7/15 | Loss: 0.3756 | Train Acc: 86.13%
Epoch 8/15 | Loss: 0.3784 | Train Acc: 86.13%
Epoch 9/15 | Loss: 0.3779 | Train Acc: 86.21%
Epoch 10/15 | Loss: 0.3653 | Train Acc: 86.46%
Epoch 11/15 | Loss: 0.3800 | Train Acc: 86.08%
Epoch 12/15 | Loss: 0.3850 | Train Acc: 85.84%
Epoch 13/15 | Loss: 0.3754 | Train Acc: 86.11%
Epoch 14/15 | Loss: 0.3772 | Train Acc: 86.09%
Epoch 15/15 | Loss: 0.3795 | Train Acc: 85.92%

Running final CNN test evaluation...
Final CNN Test Accuracy: 86.33% (Target: >= 91%)[cite: 1]


In [4]:
# --- Upgraded CNN Architecture for >= 91% Accuracy ---
class SimpleCNN:

  def __init__(self):
    # Upgraded channels: 1 -> 32 -> 64
    self.conv1 = Conv2D(
        in_channels=1, out_channels=32, kernel_size=3, padding=1
    )
    self.bn1 = BatchNorm(channels=32)
    self.pool1 = MaxPool(size=2, stride=2)  # Output: (N, 32, 14, 14)

    self.conv2 = Conv2D(
        in_channels=32, out_channels=64, kernel_size=3, padding=1
    )
    self.bn2 = BatchNorm(channels=64)
    self.pool2 = MaxPool(size=2, stride=2)  # Output: (N, 64, 7, 7)

    # Fully Connected Layers (64 * 7 * 7 = 3136 features)
    self.fc1 = Linear(64 * 7 * 7, 128)
    self.dropout = Dropout(p=0.2)  # Slightly reduced dropout
    self.fc2 = Linear(128, 10)

  def set_train_mode(self, mode):
    self.bn1.train_mode = mode
    self.bn2.train_mode = mode
    self.dropout.train_mode = mode

  def __call__(self, x):
    out = self.conv1(x)
    out = self.bn1(out)
    out = out.relu()
    out = self.pool1(out)

    out = self.conv2(out)
    out = self.bn2(out)
    out = out.relu()
    pool_out = self.pool2(out)

    N = pool_out.data.shape[0]
    out_flat = Tensor(
        pool_out.data.reshape(N, -1), parents=(pool_out,), creator="flatten"
    )

    def grad_fn():
      pool_out.grad += out_flat.grad.reshape(pool_out.data.shape)

    out_flat.grad_fn = grad_fn

    fc_out = self.fc1(out_flat).relu()
    fc_out = self.dropout(fc_out)
    logits = self.fc2(fc_out)
    return logits

  @property
  def params(self):
    return (
        self.conv1.params
        + self.bn1.params
        + self.conv2.params
        + self.bn2.params
        + self.fc1.params
        + self.fc2.params
    )


# --- Train with the Upgraded Model ---
np.random.seed(42)
cnn_model = SimpleCNN()
optimizer = Adam(cnn_model.params, lr=0.001)

batch_size = 64
epochs = 15

print("Starting Upgraded CNN Training...")
for epoch in range(epochs):
  cnn_model.set_train_mode(True)
  indices = np.arange(X_train_cnn.shape[0])
  np.random.shuffle(indices)
  X_shuff = X_train_cnn[indices]
  y_shuff = y_train[indices]

  total_loss = 0
  correct = 0

  for i in range(0, len(X_train_cnn), batch_size):
    xb = Tensor(X_shuff[i : i + batch_size])
    yb = y_shuff[i : i + batch_size]

    logits = cnn_model(xb)
    loss, probs = softmax_cross_entropy(logits, yb)

    for p in cnn_model.params:
      p.grad = np.zeros_like(p.data)

    loss.backward()
    optimizer.step()

    total_loss += loss.data * len(yb)
    preds = np.argmax(probs, axis=1)
    correct += np.sum(preds == yb)

  train_acc = correct / len(X_train_cnn)
  print(
      f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(X_train_cnn):.4f} |"
      f" Train Acc: {train_acc*100:.2f}%"
  )

# --- Final Evaluation ---
cnn_model.set_train_mode(False)
test_correct = 0

print("\nRunning final test evaluation...")
for i in range(0, len(X_test_cnn), batch_size):
  xb = Tensor(X_test_cnn[i : i + batch_size])
  yb = y_test[i : i + batch_size]

  logits = cnn_model(xb)
  exp_probs = np.exp(logits.data - np.max(logits.data, axis=1, keepdims=True))
  probs = exp_probs / np.sum(exp_probs, axis=1, keepdims=True)

  preds = np.argmax(probs, axis=1)
  test_correct += np.sum(preds == yb)

cnn_test_acc = (test_correct / len(X_test_cnn)) * 100
print(
    f"Final Upgraded CNN Test Accuracy: {cnn_test_acc:.2f}% (Target: >="
    f" 91%)[cite: 1]"
)

NameError: name 'np' is not defined

In [5]:
# --- Upgraded CNN Architecture for >= 91% Accuracy ---
class SimpleCNN:

  def __init__(self):
    # Upgraded channels: 1 -> 32 -> 64
    self.conv1 = Conv2D(
        in_channels=1, out_channels=32, kernel_size=3, padding=1
    )
    self.bn1 = BatchNorm(channels=32)
    self.pool1 = MaxPool(size=2, stride=2)  # Output: (N, 32, 14, 14)

    self.conv2 = Conv2D(
        in_channels=32, out_channels=64, kernel_size=3, padding=1
    )
    self.bn2 = BatchNorm(channels=64)
    self.pool2 = MaxPool(size=2, stride=2)  # Output: (N, 64, 7, 7)

    # Fully Connected Layers (64 * 7 * 7 = 3136 features)
    self.fc1 = Linear(64 * 7 * 7, 128)
    self.dropout = Dropout(p=0.2)  # Slightly reduced dropout
    self.fc2 = Linear(128, 10)

  def set_train_mode(self, mode):
    self.bn1.train_mode = mode
    self.bn2.train_mode = mode
    self.dropout.train_mode = mode

  def __call__(self, x):
    out = self.conv1(x)
    out = self.bn1(out)
    out = out.relu()
    out = self.pool1(out)

    out = self.conv2(out)
    out = self.bn2(out)
    out = out.relu()
    pool_out = self.pool2(out)

    N = pool_out.data.shape[0]
    out_flat = Tensor(
        pool_out.data.reshape(N, -1), parents=(pool_out,), creator="flatten"
    )

    def grad_fn():
      pool_out.grad += out_flat.grad.reshape(pool_out.data.shape)

    out_flat.grad_fn = grad_fn

    fc_out = self.fc1(out_flat).relu()
    fc_out = self.dropout(fc_out)
    logits = self.fc2(fc_out)
    return logits

  @property
  def params(self):
    return (
        self.conv1.params
        + self.bn1.params
        + self.conv2.params
        + self.bn2.params
        + self.fc1.params
        + self.fc2.params
    )


# --- Train with the Upgraded Model ---
np.random.seed(42)
cnn_model = SimpleCNN()
optimizer = Adam(cnn_model.params, lr=0.001)

batch_size = 64
epochs = 15

print("Starting Upgraded CNN Training...")
for epoch in range(epochs):
  cnn_model.set_train_mode(True)
  indices = np.arange(X_train_cnn.shape[0])
  np.random.shuffle(indices)
  X_shuff = X_train_cnn[indices]
  y_shuff = y_train[indices]

  total_loss = 0
  correct = 0

  for i in range(0, len(X_train_cnn), batch_size):
    xb = Tensor(X_shuff[i : i + batch_size])
    yb = y_shuff[i : i + batch_size]

    logits = cnn_model(xb)
    loss, probs = softmax_cross_entropy(logits, yb)

    for p in cnn_model.params:
      p.grad = np.zeros_like(p.data)

    loss.backward()
    optimizer.step()

    total_loss += loss.data * len(yb)
    preds = np.argmax(probs, axis=1)
    correct += np.sum(preds == yb)

  train_acc = correct / len(X_train_cnn)
  print(
      f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(X_train_cnn):.4f} |"
      f" Train Acc: {train_acc*100:.2f}%"
  )

# --- Final Evaluation ---
cnn_model.set_train_mode(False)
test_correct = 0

print("\nRunning final test evaluation...")
for i in range(0, len(X_test_cnn), batch_size):
  xb = Tensor(X_test_cnn[i : i + batch_size])
  yb = y_test[i : i + batch_size]

  logits = cnn_model(xb)
  exp_probs = np.exp(logits.data - np.max(logits.data, axis=1, keepdims=True))
  probs = exp_probs / np.sum(exp_probs, axis=1, keepdims=True)

  preds = np.argmax(probs, axis=1)
  test_correct += np.sum(preds == yb)

cnn_test_acc = (test_correct / len(X_test_cnn)) * 100
print(
    f"Final Upgraded CNN Test Accuracy: {cnn_test_acc:.2f}% (Target: >="
    f" 91%)[cite: 1]"
)

NameError: name 'np' is not defined